In [2]:
import nvdlib
import json
import pandas as pd
import numpy as np
import re
import ast
from tqdm.notebook import tqdm
import sys
import string
import nltk
from nltk.tokenize import word_tokenize
import os
import fnmatch
import warnings
from pprint import pprint
warnings.filterwarnings('ignore')

In [9]:
from nltk.corpus import stopwords
tqdm.pandas()
directory_path = "/home/umd-user/Desktop/navex_project/navex_tests/phpBB-2.0.23"
appName = 'phpbb'
extension = ''
appVersion = '2.0.23'

nltk.download('stopwords')
nltk.download('punkt')

stopwords = list(filter(lambda x: len(x)>1, stopwords.words('english')))
punctuation = set(string.punctuation)
punctuation.remove('(')
punctuation.remove(')')
punctuation.remove('$')

[nltk_data] Downloading package stopwords to /home/umd-
[nltk_data]     user/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/umd-user/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [4]:
def getFiles(description):
    # extensions = ['php', 'html', 'js']
    extensions = ['php']
    result = []
    for ext in extensions:
        result += re.findall(r'\b\w+\.' + ext + r'\b', description, re.IGNORECASE)
    return result

def getVersions(description):
    result = re.findall(r'\d+\.\d+\.\d+', description)
    for version in result: description = description.replace(version, '')
    result += re.findall(r'\d+\.\d+', description)
    if result==[]:
        return ['0']
    else: return result

def getVulnerability(description):
    desc = description.lower()
    if "sql" in desc:
        return "SQL Injection"
    elif "xss" in desc or "cross-site scripting" in desc or "cross site scripting" in desc:
        return "XSS"
    elif "file upload" in desc or "file inclusion" in desc:
        return "File Inclusion"
    elif "file access" in desc:
        return "File Access"
    elif "session" in desc:
        return "Session Fixation"
    elif "code injection" in desc:
        return "Code Injection"
    elif "command" in desc:
        return "Command Execution"
    # elif "csrf" in desc or "request forgery" in desc:
    #     return "CSRF"
    else:
        return "NA"
    
# Compute whether appVersion was released before cveVersion (inclusive)
def compareVersions(appVersion, cveVersions):
    flag = False
    if (len(cveVersions)==0 or len(appVersion)==0 or cveVersions==['0']): flag = True
    for cveVersion in cveVersions:
        v1 = list(map(int, appVersion.split('.')))
        v2 = list(map(int, cveVersion.split('.')))
        size = min(len(v1), len(v2))
        v1 = v1[:size]
        v2 = v2[:size]
        if (v1 <= v2): flag = True
    return flag

def getParameters(description):
    words = word_tokenize(description)
    words = [word for word in words if word.lower() not in stopwords and word not in punctuation]
    indices = [i for i in range(1, len(words)) if words[i] == "parameter" or words[i] == "parameters"]
    params = [words[i-1] for i in indices]
    indices = [i for i in range(1, len(words)-2) if (words[i-1]=='(' and words[i].isdigit() and words[i+1]==')')]
    params += [words[i+2] for i in indices]
    indices = [i for i in range(len(words)-1) if words[i] == "function" or words[i]=="$"]
    params += [words[i+1] for i in indices]
    params = [param.replace('"', '') for param in params if ('.php' not in param and '/' not in param)]
    return list(set(params))

def getCVEFromNavex(cve, file):
    flag = False
    for index, row in cve.iterrows():
        for fileName in row['filenames']:
            if fileName in file:
                flag = True
                cve_id = row['id']
                break
    if not flag: cve_id = 'NA'
    return cve_id

def filterOnFiles(fileNames):
    if fileNames == []: return True
    for root, dirs, files in os.walk(directory_path):
        for fileName in fileNames:
            for filename in fnmatch.filter(files, fileName):
                # file_path = os.path.join(root, filename)
                return True
    return False

def getFilesFromParameters(parameters):
    file_paths = []
    pattern = r'(?:' + '|'.join(re.escape(parameter) for parameter in parameters) + r')\b'
    if parameters == []: return file_paths
    for root, dirs, files in os.walk(directory_path):
        for filename in files:
            file_path = os.path.join(root, filename)
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as file:
                content = file.read()
                if re.search(pattern, content):
                    file_paths.append(file_path)
    return list(set(file_paths))

In [5]:
r = nvdlib.searchCVE(keywordSearch=appName)
jsonFormattedCVE = json.dumps(ast.literal_eval(str(r)))

In [19]:
cve = pd.read_json(jsonFormattedCVE)

cve['descriptions'] = cve['descriptions'].apply(lambda descriptions: list(filter(lambda x: x["lang"]=="en", descriptions)))
cve['descriptions'] = cve['descriptions'].apply(lambda descriptions: descriptions[0]['value'])
cve['versions'] = cve['descriptions'].apply(getVersions)
cve['filenames'] = cve['descriptions'].apply(getFiles)
cve['cve_vulnerability'] = cve['descriptions'].apply(getVulnerability)
cve['parameters'] = cve['descriptions'].apply(getParameters)
cve['relevant_version'] = cve['versions'].apply(lambda x: compareVersions(appVersion, x) and compareVersions(x[0], ['2.9']))

cve = cve[cve['cve_vulnerability']!='NA']
cve = cve[cve['relevant_version']==True]

cve = cve[cve['filenames'].apply(filterOnFiles)]
cve = cve[cve['parameters'].apply(lambda params: getFilesFromParameters(params) != [])]
cve = cve[cve['filenames'].map(lambda x: x!=[]) | cve['parameters'].map(lambda x: x!=[])]
# cve['filenames'] = cve.apply(lambda x: x.filenames if x.filenames!=[] else list(map(lambda x: x.split('/')[-1], getFilesFromParameters(x.parameters))), axis=1)

cve = cve.loc[:, ['id', 'cve_vulnerability', 'versions', 'filenames', 'parameters', 'descriptions']]

cve.to_excel(f"/home/umd-user/Desktop/navex_project/navex_utils/cve/{appName}-{extension}cve.xlsx")
cve.shape

(12, 6)

In [20]:
cve

,id,cve_vulnerability,versions,filenames,parameters,descriptions
14,CVE-2003-0484,XSS,[0],[viewtopic.php],[topic_id],Cross-site scripting (XSS) vulnerability in vi...
18,CVE-2003-1244,SQL Injection,"[2.0.1, 2.0.2, 2.0]","[page_header.php, index.php]",[forum_id],SQL injection vulnerability in page_header.php...
87,CVE-2006-1895,Code Injection,[0],"[template.php, bbcode.php]","[bypasses, used]",Direct static code injection vulnerability in ...
88,CVE-2006-1896,Code Injection,[0],[],[theme],Unspecified vulnerability in phpBB allows remo...
94,CVE-2006-2283,File Inclusion,"[2.9.5, 3.0]","[auth.php, auth.php]","[auth_SMF, smf_root_path, phpbb_root_path, aut...",Multiple PHP remote file inclusion vulnerabili...
102,CVE-2006-2828,File Inclusion,[0],"[index.php, admin_ug_auth.php, admin_board.php...","[executed, phpbb_root_path]",Global variable overwrite vulnerability in PHP...
103,CVE-2006-2865,File Inclusion,[0],"[template.php, template.php]",[page],PHP remote file inclusion vulnerability in tem...
141,CVE-2006-5387,File Inclusion,[0],[constants.php],[phpbb_root_path],PHP remote file inclusion vulnerability in mod...
165,CVE-2007-0680,File Inclusion,[0],[functions.php],[phpbb_root_path],PHP remote file inclusion vulnerability in inc...
169,CVE-2007-0762,File Inclusion,[0],[functions.php],[phpbb_root_path],PHP remote file inclusion vulnerability in inc...


In [21]:
# Output of the navex joern extension
navex = pd.read_json(f'/home/umd-user/Desktop/navex_project/navex_utils/paths/{appName}-{extension}output.json')
navex.shape

(257, 8)

In [22]:
def getCVEidsFromPath(row, pathRow):
    flag = False
    cve_id = np.nan
    if type(row['filenames'])!=list: files = ast.literal_eval(row['filenames'])
    else: files = row['filenames']
    if type(row['parameters'])!=list: cveParams = ast.literal_eval(row['parameters'])
    else: cveParams = row['parameters']
    if not files: files = ['']
    
    for fileName in files:
        if row['cve_vulnerability'] == pathRow['vulnerability']: 
            if not cveParams and fileName in pathRow.loc['filename']:
                flag = True
            else:
                for param in cveParams:
                    if (fileName in pathRow.loc['filename']) or ((('$' + param.lower().replace('$','')) in pathRow['code'].lower()) or (param == pathRow['methodname'])):
                        flag = True
            if flag:
                    cve_id = row['id']
    return cve_id

In [24]:
CVE_ids = navex.progress_apply(lambda navexRow: list(cve.apply(lambda x: getCVEidsFromPath(x, navexRow), axis=1).dropna().unique()), axis=1)
navex['CVE_ids'] = CVE_ids
navex.head()

  0%|          | 0/257 [00:00<?, ?it/s]

,pathid,vulnerability,nodeid,methodname,filename,linenumber,code,sanitized,CVE_ids
0,1,XSS,3649,<global>,phpBB-2.0.23/phpBB2/admin/admin_db_utilities.php,700,"$HTTP_POST_VARS[""backup_type""]",FALSE,[]
1,1,XSS,3644,<global>,phpBB-2.0.23/phpBB2/admin/admin_db_utilities.php,700,"isset($HTTP_POST_VARS[""backup_type""]) ? $HTTP_...",FALSE,[]
2,1,XSS,2020,<global>,phpBB-2.0.23/phpBB2/admin/admin_db_utilities.php,700,$backup_type,FALSE,[]
3,1,XSS,2022,<global>,phpBB-2.0.23/phpBB2/admin/admin_db_utilities.php,816,$backup_type,FALSE,[]
4,1,XSS,4040,<global>,phpBB-2.0.23/phpBB2/admin/admin_db_utilities.php,816,"pg_get_sequences(""\n"",$backup_type)",FALSE,[]


In [25]:
# Check rows that matched with a CVE
emptyList = pd.Series([np.nan] * len(navex['CVE_ids'])).fillna('[]')
navex[navex['CVE_ids'].astype(str) != emptyList].explode('CVE_ids').head()

,pathid,vulnerability,nodeid,methodname,filename,linenumber,code,sanitized,CVE_ids
93,1,File Inclusion,86555,<global>,phpBB-2.0.23/phpBB2/install/install.php,467,"$phpbb_root_path . ""language/lang_"" . $language",FALSE,CVE-2006-2283
93,1,File Inclusion,86555,<global>,phpBB-2.0.23/phpBB2/install/install.php,467,"$phpbb_root_path . ""language/lang_"" . $language",FALSE,CVE-2006-2828
93,1,File Inclusion,86555,<global>,phpBB-2.0.23/phpBB2/install/install.php,467,"$phpbb_root_path . ""language/lang_"" . $language",FALSE,CVE-2006-5387
93,1,File Inclusion,86555,<global>,phpBB-2.0.23/phpBB2/install/install.php,467,"$phpbb_root_path . ""language/lang_"" . $language",FALSE,CVE-2007-0680
93,1,File Inclusion,86555,<global>,phpBB-2.0.23/phpBB2/install/install.php,467,"$phpbb_root_path . ""language/lang_"" . $language",FALSE,CVE-2007-0762


In [26]:
navex = navex.explode('CVE_ids')
navex.shape

(357, 9)

In [27]:
navex = navex.merge(cve, how='left', left_on='CVE_ids', right_on='id').drop(columns=['id', 'cve_vulnerability'])

In [28]:
matchedCVEs = {e for e in navex['CVE_ids'].dropna()}
print("Number of exploit matches:", len(matchedCVEs), "out of #" + str(len(cve['id'])), "CVEs")
pprint(matchedCVEs)

Number of exploit matches: 7 out of #12 CVEs
{'CVE-2006-2283',
 'CVE-2006-2828',
 'CVE-2006-5387',
 'CVE-2006-7174',
 'CVE-2007-0680',
 'CVE-2007-0762',
 'CVE-2007-4984'}


In [120]:
# Save data to excel
dfToSave = navex.dropna()
dfToSave.reset_index(inplace=True)
dfToSave.to_excel(f"/home/umd-user/Desktop/navex_project/navex_utils/cve/{appName}-{extension}navex.xlsx")
dfToSave.shape

(45772, 14)

In [121]:
# xssPaths = navex[navex['vulnerability'] == 'XSS']
xssPaths = navex
acrossDb = xssPaths[xssPaths['code'].apply(lambda x: '_query' in x)]
notDb = xssPaths[xssPaths['pathid'].apply(lambda x: x not in acrossDb['pathid'])]
acrossDb['CVE_ids'].unique().shape, notDb['CVE_ids'].unique().shape, xssPaths['CVE_ids'].unique().shape

((3,), (4,), (4,))